# BugTwin | Duplicate Bug Report Finder

A small NLP application that retrieves similar software bug reports using **TF-IDF** and **MiniLM sentence embeddings**. Enter a bug description, choose a method, and review the five closest reports.

**Data:** 30 fictional but realistic issue reports written for this demo, plus six manually labeled example queries. Evaluation scores are illustrative, **not** results on a real bug-tracking dataset. The search logic can be used with real issue records in the same `id`, `title`, `description` format.

**Run:** Google Colab → **Runtime → Run all**. The final cell opens the Gradio application. MiniLM downloads once from Hugging Face; no paid API is required.

In [ ]:
import os
import subprocess
import sys

if os.getenv("BUGTWIN_SKIP_INSTALL") != "1":
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "sentence-transformers>=3.3,<6", "gradio>=5,<7",
    ])

## 1. Demo bug reports

Each record has a stable ID, a short summary, and a description. The embedded corpus makes the notebook runnable without external dataset files.

In [ ]:
from io import StringIO
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA = """id|title|description
BT-101|Cannot sign in after password reset|The login form rejects valid credentials after a password change.
BT-102|Password recovery email missing|Reset email never arrives in the inbox after submitting the recovery request.
BT-103|Login screen freezes after changing password|Authentication page becomes unresponsive and cannot be clicked after resetting the account password.
BT-104|Session expires immediately after login|A newly authenticated session ends within a few seconds and returns to sign in.
BT-105|Two factor authentication code rejected|A correct one time login code is repeatedly marked invalid.
BT-201|Export button does nothing|Clicking the export action in the toolbar does not create a file.
BT-202|Large CSV export produces empty file|Exporting a large table creates a zero byte CSV file.
BT-203|PDF report export loses images|Downloaded PDF contains text but drops embedded diagrams.
BT-204|Download filename contains strange characters|Generated report filename contains corrupted Unicode symbols.
BT-205|Spreadsheet export skips final row|The last record is missing when downloading a table as XLSX.
BT-301|Editor cursor jumps while typing|Text caret unexpectedly moves to the start of the line during input.
BT-302|Undo stops working after paste|After pasting multiline text the editor undo shortcut no longer changes the content.
BT-303|Auto save overwrites recent edits|The editor restores an older document version and loses recent typing.
BT-304|Syntax highlighting disappears in Python files|Code coloring stops after changing the editor language to Python.
BT-305|Search and replace removes extra spaces|Replacing text unexpectedly deletes whitespace between tokens.
BT-401|Application stuck on splash screen|Desktop client remains on startup logo and never opens the workspace.
BT-402|Startup crashes after application update|The app closes immediately after installing the latest version.
BT-403|Blank window displayed on first launch|Opening the application shows a white screen with no controls.
BT-404|Plugin causes slow startup|Loading third party extensions makes the program take several minutes to open.
BT-405|Cannot reopen previous project at startup|Recent project selection fails with a missing workspace error.
BT-501|Connection times out behind firewall|The client cannot reach the service when corporate firewall rules are enabled.
BT-502|Proxy connection fails certificate check|Connecting through an HTTP proxy triggers an SSL certificate verification error.
BT-503|WebSocket disconnects repeatedly|Real time updates stop because the socket connection closes every few seconds.
BT-504|Uploads fail on unstable network|Sending a file returns a network error after a brief WiFi interruption.
BT-505|Offline mode does not synchronize changes|Edits made without internet access never sync when the connection returns.
BT-601|Dark theme resets after restart|After restarting the program the selected dark mode changes back to light mode.
BT-602|Notification sounds continue when muted|Alert sound plays even though notification audio is disabled in settings.
BT-603|Language preference not saved|Choosing another interface language reverts to English at next launch.
BT-604|Keyboard shortcut setting ignored|Custom key bindings are not applied after saving preferences.
BT-605|Font size preference does not persist|Editor text size returns to the default after closing the app.
"""

reports = pd.read_csv(StringIO(DATA), sep="|")
texts = (reports["title"] + ". " + reports["description"]).tolist()
print(f"Loaded {len(reports)} example bug reports")
print(reports.head(3).to_string(index=False))

## 2. NLP representations

**TF-IDF** represents each document using weighted word and phrase frequencies. **MiniLM** maps each report into a dense semantic vector. Both methods rank candidates using cosine similarity. These are relevance scores, not verified probabilities of duplication.

In [ ]:
tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(texts)

model, vectors = None, None
if os.getenv("BUGTWIN_CI") != "1":
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        vectors = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
        print("MiniLM embeddings ready:", vectors.shape)
    except Exception as exc:
        print(f"MiniLM unavailable ({type(exc).__name__}); TF-IDF search remains available.")

## 3. Search

The application ranks historical records by similarity and returns the top five potential matches for human review.

In [ ]:
def find_similar(title: str, description: str, method: str = "MiniLM", top_k: int = 5) -> pd.DataFrame:
    query = f"{title or ''} {description or ''}".strip()
    columns = ["id", "similarity", "title", "description"]
    if not query:
        return pd.DataFrame(columns=columns)
    if len(query) > 3000:
        raise ValueError("Please keep the report under 3,000 characters.")

    if method == "TF-IDF":
        scores = cosine_similarity(tfidf.transform([query]), tfidf_matrix).ravel()
    elif method == "MiniLM" and model is not None:
        query_vector = model.encode([query], normalize_embeddings=True)[0]
        scores = vectors @ query_vector
    else:
        raise ValueError(f"Unavailable retrieval method: {method}")

    positions = np.argsort(-scores, kind="stable")[:int(top_k)]
    matches = reports.iloc[positions][["id", "title", "description"]].copy()
    matches.insert(1, "similarity", np.round(scores[positions], 3))
    return matches.reset_index(drop=True)[columns]

assert reports["id"].is_unique and reports["title"].notna().all()
assert len(find_similar("login password reset", "", "TF-IDF")) == 5
print("Basic checks passed")

## 4. Small evaluation

Six manually prepared queries have one intended reference match each. **Hit@3** checks whether that report appears among the top three. These examples help compare methods; they do not measure real-world accuracy.

In [ ]:
examples = [
    ("Authentication page freezes after resetting password", "BT-103"),
    ("Saving a big table gives an empty CSV export", "BT-202"),
    ("Undo shortcut fails after pasting multiple lines", "BT-302"),
    ("Program hangs at the opening splash logo", "BT-401"),
    ("SSL verification failure when using proxy", "BT-502"),
    ("Dark mode changes to light after app restart", "BT-601"),
]
methods = ["TF-IDF"] + (["MiniLM"] if model is not None else [])
summary = []
for method in methods:
    hits = sum(target in find_similar(query, "", method, 3)["id"].tolist()
               for query, target in examples)
    summary.append({"method": method, "Hit@3": f"{hits}/{len(examples)}"})
print(pd.DataFrame(summary).to_string(index=False))

## 5. Gradio application

Try the included example or describe a new defect. Results are *potential* duplicates; similarity alone cannot establish a shared root cause.

The Gradio share link is publicly reachable while the session is running. **Do not enter confidential bug reports.**

In [ ]:
if os.getenv("BUGTWIN_CI") != "1":
    import gradio as gr

    available = ["TF-IDF"] + (["MiniLM"] if model is not None else [])

    def run_search(title: str, description: str, method: str):
        try:
            matches = find_similar(title, description, method, 5)
            if matches.empty:
                return matches, "Enter a title or description to search."
            return matches, f"Found {len(matches)} candidate reports. Review them before marking duplicates."
        except ValueError as error:
            return pd.DataFrame(columns=["id", "similarity", "title", "description"]), str(error)

    with gr.Blocks(title="BugTwin") as demo:
        gr.Markdown("# BugTwin\nFind related software bug reports using NLP.")
        with gr.Row():
            title = gr.Textbox(label="Bug title", placeholder="Login freezes after password reset")
            method = gr.Radio(available, value=available[-1], label="Search method")
        description = gr.Textbox(label="Bug description", lines=3)
        button = gr.Button("Find potential duplicates", variant="primary")
        output = gr.Dataframe(label="Top 5 candidates", interactive=False, wrap=True)
        status = gr.Markdown()
        button.click(run_search, [title, description, method], [output, status])
        gr.Examples([[query, ""] for query, _ in examples[:3]], inputs=[title, description])
        gr.Markdown("Demo data is fictional. Similarity scores are not duplicate probabilities.")

    demo.launch(share=True)